In [3]:
import pandas as pd
import sqlite3

# create database
conn = sqlite3.connect("ecommerce.db")





In [4]:
# load csv files
customers = pd.read_csv("customers.csv",encoding="latin1")
orders = pd.read_csv("orders.csv",encoding="latin1")
products = pd.read_csv("products.csv",encoding="latin1")
sellers = pd.read_csv("sellers.csv",encoding="latin1")
payments = pd.read_csv("payments.csv",encoding="latin1")
order_reviews = pd.read_csv("order_reviews.csv",encoding="latin1")
order_items = pd.read_csv("order_items.csv",encoding="latin1")
geolocation = pd.read_csv("geolocation.csv",encoding="latin1")

In [5]:
# convert to SQL tables
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
sellers.to_sql("sellers", conn, if_exists="replace", index=False)
payments.to_sql("payments", conn, if_exists="replace", index=False)
order_reviews.to_sql("order_reviews", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
geolocation.to_sql("geolocation", conn, if_exists="replace", index=False)

print("Tables created successfully")

Tables created successfully


In [6]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [7]:
import pandas as pd

In [8]:
#1.2.Count the Cities & States of customers who ordered during the given period.
query="""
select customer_state ,count(*) as co
from customers
group by customer_state
order by co desc"""


pd.read_sql(query,conn)

,customer_state,co
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


In [9]:
#--In-depth Exploration:
#--Is there a growing trend in the no. of orders placed over the past years?

query1="""
with orders_years as ( 
  select order_id,strftime('%Y', order_purchase_timestamp) as year_order
  from orders
)
select year_order,count(distinct order_id) as orders_count
from orders_years
group by year_order
order by year_order asc
"""
pd.read_sql(query1,conn)


,year_order,orders_count
0,2016,329
1,2017,45101
2,2018,54011


In [10]:
#--Can we see some kind of monthly seasonality in terms of the no. of orders being placed?

query2=""" 
with orders_years_month as ( 
  select order_id,
  strftime('%Y', order_purchase_timestamp) as year_order,
  strftime('%m' ,order_purchase_timestamp ) as month_order
  from orders
)
select month_order,
count(distinct case when year_order = '2016' then order_id end)  as orders_2016,
count(distinct case when year_order = '2017' then order_id end)  as orders_2017,
count(distinct case when year_order = '2018' then order_id end)  as orders_2018
from orders_years_month
group by month_order
order by month_order asc
"""

pd.read_sql(query2,conn)

,month_order,orders_2016,orders_2017,orders_2018
0,01,0,800,7269
1,02,0,1780,6728
2,03,0,2682,7211
3,04,0,2404,6939
4,05,0,3700,6873
5,06,0,3245,6167
6,07,0,4026,6292
7,08,0,4331,6512
8,09,4,4285,16
9,10,324,4631,4


In [11]:
#--Evolution of E-commerce orders in the Brazil region:
#--Get the month on month no. of orders placed in each state.

query3="""with cte as (
  select o.order_id, o.customer_id,c.customer_city,c.customer_state,
  strftime("%Y" ,o.order_purchase_timestamp)as year,
  strftime("%m",o.order_purchase_timestamp) as month
  from orders as o 
  inner join customers as c 
  on o.customer_id = c.customer_id
)
select customer_state,year,
count(distinct case when month = "01" then order_id end) as month_1, 
count(distinct case when month = "02" then order_id end) as month_2,
count(distinct case when month = "03" then order_id end) as month_3,
count(distinct case when month = "04" then order_id end) as month_4,
count(distinct case when month = "05" then order_id end) as month_5,
count(distinct case when month = "06" then order_id end) as month_6,
count(distinct case when month = "07" then order_id end) as month_7,
count(distinct case when month = "08" then order_id end) as month_8,
count(distinct case when month = "09" then order_id end) as month_9,
count(distinct case when month = "10" then order_id end) as month_10,
count(distinct case when month = "11" then order_id end) as month_11,
count(distinct case when month = "12" then order_id end) as month_12
from cte
group by customer_state,year
order by customer_state,year
"""

pd.read_sql(query3,conn)

,customer_state,year,month_1,month_2,month_3,month_4,month_5,month_6,month_7,month_8,month_9,month_10,month_11,month_12
0,AC,2017,2,3,2,5,8,4,5,4,5,6,5,5
1,AC,2018,6,3,2,4,2,3,4,3,0,0,0,0
2,AL,2016,0,0,0,0,0,0,0,0,0,2,0,0
3,AL,2017,2,12,10,23,27,10,17,18,20,28,26,14
4,AL,2018,37,27,30,28,19,24,23,16,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,SP,2016,0,0,0,0,0,0,0,0,2,113,0,0
71,SP,2017,299,654,1010,908,1425,1331,1604,1729,1638,1793,3012,2357
72,SP,2018,3052,2703,3037,3059,3207,2773,2777,3253,8,2,0,0
73,TO,2017,2,7,8,14,18,8,1,15,17,13,17,14


In [12]:
#--How are the customers distributed across all the states?

query4 = """with cte as (
  select o.order_id, o.customer_id,c.customer_city,c.customer_state,
  strftime("%Y", o.order_purchase_timestamp)as year,
  strftime("%m",o.order_purchase_timestamp) as month
  from orders as o 
  inner join customers as c 
  on o.customer_id = c.customer_id
)
select customer_state ,count(distinct customer_id) as customers_count 
from cte
group by customer_state
order by customers_count desc
"""
pd.read_sql(query4,conn)

,customer_state,customers_count
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045
5,SC,3637
6,BA,3380
7,DF,2140
8,ES,2033
9,GO,2020


In [13]:

#--Impact on Economy: Analyze the money movement by e-commerce by looking at order prices, freight and others.
#--Get the % increase in the cost of orders from year 2017 to 2018 (include months between Jan to Aug only).
#--You can use the "payment_value" column in the payments table to get the cost of orders.

query5 = """with cte as (
  select o.order_id, o.customer_id,
  strftime("%Y" ,o.order_purchase_timestamp)as year,
  strftime("%m",o.order_purchase_timestamp) as month,
  p.payment_value
  from orders as o 
  inner join payments as p 
  on o.order_id = p.order_id
),cte1 as(
select year,sum(payment_value) as total_payment_value
from cte
where month in ("01","02","03","04","05","06","07","08") and year in ("2017","2018")
group by year)

select  round((((select total_payment_value from cte1 where year="2018") - (select total_payment_value from cte1 where year="2017"))*100 / (select total_payment_value from cte1 where year="2017")),2) as percentage1 
from cte1 
limit 1
"""
pd.read_sql(query5,conn)

,percentage1
0,136.98


In [14]:
#--Calculate the Total & Average value of order price for each state.


query6 = """with order_price_freight as (
  select order_id,sum(price) as price_s ,
  sum(freight_value) as freight_value_s
  from order_items
  group by order_id
),
cte1 as (
select c.customer_state,sum(opf.price_s) as total_orders_price , avg(opf.price_s) as average_orders_price
from customers as c
inner join orders as o 
on c.customer_id = o.customer_id 
inner join order_price_freight as opf 
on o.order_id = opf.order_id 
group by c.customer_state)
select *
from cte1
order by total_orders_price desc
"""
pd.read_sql(query6,conn)

,customer_state,total_orders_price,average_orders_price
0,SP,5.202955e+06,125.751179
1,RJ,1.824093e+06,142.931568
2,MG,1.585308e+06,137.327445
3,RS,7.503040e+05,138.126661
4,PR,6.830838e+05,136.671421
5,SC,5.205533e+05,144.117757
6,BA,5.113500e+05,152.278139
7,DF,3.026039e+05,142.401854
8,GO,2.945919e+05,146.782237
9,ES,2.750373e+05,135.820894


In [15]:
#--Calculate the Total & Average value of order freight for each state.



query7 = """with order_price_freight as (
  select order_id,sum(price) as price_s ,
  sum(freight_value) as freight_value_s
  from order_items
  group by order_id
),
cte1 as (
select c.customer_state,sum(opf.freight_value_s) as total_orders_freight_price , avg(opf.freight_value_s) as average_orders_freight_price
from customers as c
inner join orders as o 
on c.customer_id = o.customer_id 
inner join order_price_freight as opf 
on o.order_id = opf.order_id 
group by c.customer_state)
select *
from cte1
order by total_orders_freight_price desc
"""
pd.read_sql(query7,conn)

,customer_state,total_orders_freight_price,average_orders_freight_price
0,SP,718723.07,17.370950
1,RJ,305589.31,23.945252
2,MG,270853.46,23.462704
3,RS,135522.74,24.948958
4,PR,117851.68,23.579768
5,BA,100156.68,29.826289
6,SC,89660.26,24.822885
7,PE,59449.66,36.073823
8,GO,53114.98,26.464863
9,DF,50625.50,23.823765


In [16]:
#--Find the no. of days taken to deliver each order from the order’s purchase date as delivery time.
#--Also, calculate the difference (in days) between the estimated & actual delivery date of an order.
#--Do this in a single query.



query8 = """with cte as (
  select order_id,julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp) as time_to_deliver,
  julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date) as diff_estimated_delivery
  from orders
)
select * 
from cte
where time_to_deliver is not null and diff_estimated_delivery is not null
"""

pd.read_sql(query8,conn)

,order_id,time_to_deliver,diff_estimated_delivery
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,-7.107488
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,-5.355729
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,-17.245498
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,-12.980069
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,-9.238171
...,...,...,...
96471,9c5dedf39a927c1b2549525ed64a053c,8.218009,-10.369433
96472,63943bddc261676b46f01ca7ac2f7bd8,22.193727,-1.265324
96473,83c1379a015df1e13d02aae0204711ab,24.859421,-5.524803
96474,11c177c8e97725db2631073c19f07b62,17.086424,-20.018819


In [17]:
#--Find out the top 5 states with the highest & lowest average delivery time.

query9 = """with cte as (
  select c.customer_state,round(avg(julianday(o.order_delivered_customer_date)- julianday(o.order_purchase_timestamp))) as avg_time_to_deliver
  from orders as o 
  inner join customers as c 
  on o.customer_id = c.customer_id 
  group by c.customer_state 
)
select * 
from cte
where avg_time_to_deliver is not null 
order by avg_time_to_deliver desc
limit 5
"""
pd.read_sql(query9,conn)

,customer_state,avg_time_to_deliver
0,RR,29.0
1,AP,27.0
2,AM,26.0
3,AL,25.0
4,PA,24.0


In [18]:

query10="""with cte as (
  select c.customer_state,round(avg(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp))) as avg_time_to_deliver
  from orders as o 
  inner join customers as c 
  on o.customer_id = c.customer_id 
  group by c.customer_state 
)
select * 
from cte
where avg_time_to_deliver is not null 
order by avg_time_to_deliver asc
limit 5
"""
pd.read_sql(query10,conn)

,customer_state,avg_time_to_deliver
0,SP,9.0
1,MG,12.0
2,PR,12.0
3,DF,13.0
4,RJ,15.0


In [19]:
#--Find out the top 5 states where the order delivery is really fast as compared to the estimated date of delivery.
#--You can use the difference between the averages of actual & estimated delivery date to figure out how fast the delivery was for each state.

query11 = """with cte as (
  select c.customer_state,round(avg(julianday(order_delivered_customer_date) - julianday(order_estimated_delivery_date))) as avg_diff_estimated_delivery
  from orders as o 
  inner join customers as c 
  on o.customer_id = c.customer_id 
  group by c.customer_state 
)
select * 
from cte
where avg_diff_estimated_delivery is not null 
order by avg_diff_estimated_delivery asc
limit 5
"""
pd.read_sql(query11,conn)

,customer_state,avg_diff_estimated_delivery
0,AC,-20.0
1,AM,-19.0
2,AP,-19.0
3,RO,-19.0
4,RR,-17.0


In [20]:
#Analysis based on the payments:
#Find the month on month no. of orders placed using different payment types.



query12="""with cte as ( 
  select strftime("%Y-%m",o.order_purchase_timestamp) as date_month,
  sum(case when p.payment_type = "credit_card" then 1 else 0 end ) as credit_card,
  sum(case when p.payment_type = "UPI" then 1 else 0 end ) as upi,
  sum(case when p.payment_type = "debit_card" then 1 else 0 end ) as debit_card,
  sum(case when p.payment_type = "voucher" then 1 else 0 end ) as voucher,
  sum(case when p.payment_type = "not_defined" then 1 else 0 end ) as not_defined
  from orders as o 
  inner join payments as p 
  on o.order_id = p.order_id
  group by strftime("%Y-%m",o.order_purchase_timestamp)
)
select * 
from cte
order by date_month asc
"""
pd.read_sql(query12,conn)

,date_month,credit_card,upi,debit_card,voucher,not_defined
0,2016-09,3,0,0,0,0
1,2016-10,254,63,2,23,0
2,2016-12,1,0,0,0,0
3,2017-01,583,197,9,61,0
4,2017-02,1356,398,13,119,0
5,2017-03,2016,590,31,200,0
6,2017-04,1846,496,27,202,0
7,2017-05,2853,772,30,289,0
8,2017-06,2463,707,27,239,0
9,2017-07,3086,845,22,364,0


In [21]:
#Find the no. of orders placed on the basis of the payment installments that have been paid.


query13="""with cte as ( 
  select p.payment_installments,count(o.order_id) as count_orders

  from orders as o 
  inner join payments as p 
  on o.order_id = p.order_id
  group by p.payment_installments
)
select * 
from cte 
order by count_orders desc"""
pd.read_sql(query13,conn)

,payment_installments,count_orders
0,1,52546
1,2,12413
2,3,10461
3,4,7098
4,10,5328
5,5,5239
6,8,4268
7,6,3920
8,7,1626
9,9,644
